In [170]:
# conda activate chronocell

import os, sys
import numpy as np
import pandas as pd

sys.path.append("/mnt/lareaulab/reliscu/programs/FGP_2024")
sys.path.append("/mnt/lareaulab/reliscu/projects/Chronocell/analyses/simulations/code")
sys.path.append("code")

import Chronocell
from reconstruct_RNA_history import *

from protein_from_RNA import *

In [2]:
# Get traj object from running Chronocell
import pickle
with open("eLNPs_var>1.2_traj_WS.pkl", "rb") as f:
    traj = pickle.load(f)

In [70]:
Y = traj.X
tau = traj.tau # State transition times (global)
t = traj.t
theta = traj.theta
topo = traj.topo

theta_ = theta.copy()
a0 = theta_[:2, 0] # Starting RNA abundance 
a = theta_[:2, 1:len(topo.flatten())] 
beta = theta_[:2, -2] # Splicing rate
alpha = a * beta[:2, None] # These values are divided by splicing rate; removing this factor now
gamma = theta_[:2, -1] # Degradation rate
state_grid = np.searchsorted(tau, t, side="left") - 1

U_max = 15
S_max = 20
states, index_for = enumerate_states(U_max, S_max)

array([48, 69, 45, ...,  1,  5,  0])

In [233]:
most_prob_t

[np.float64(3.878787878787879),
 np.float64(5.575757575757576),
 np.float64(3.6363636363636367),
 np.float64(3.6363636363636367),
 np.float64(4.363636363636364),
 np.float64(3.7979797979797985),
 np.float64(4.040404040404041),
 np.float64(1.777777777777778),
 np.float64(4.767676767676768),
 np.float64(3.555555555555556),
 np.float64(3.878787878787879),
 np.float64(4.282828282828283),
 np.float64(3.555555555555556),
 np.float64(3.7171717171717176),
 np.float64(6.060606060606061),
 np.float64(4.040404040404041),
 np.float64(4.444444444444445),
 np.float64(6.141414141414142),
 np.float64(3.7171717171717176),
 np.float64(3.6363636363636367),
 np.float64(5.333333333333334),
 np.float64(3.95959595959596),
 np.float64(4.92929292929293),
 np.float64(2.909090909090909),
 np.float64(5.6565656565656575),
 np.float64(4.606060606060606),
 np.float64(3.7979797979797985),
 np.float64(4.363636363636364),
 np.float64(3.7171717171717176),
 np.float64(4.202020202020202),
 np.float64(0.8080808080808082),


In [91]:
# Prep rate matrices

A_per_gene = [] 

for j in range(0, alpha.shape[0]):
    A_for_this_gene = []
    for i in range(0, alpha.shape[1]):
        rxns = define_reactions(alpha[j, i], beta[j], gamma[j])
        A1 = create_transition_matrix(rxns, U_max, S_max)
        A_for_this_gene.append(A1)
    A_per_gene.append(A_for_this_gene)

In [92]:
# Initialize X_fwd with stationary distribution (steady state at t=0)

pi_per_gene = []

for j in range(0, alpha.shape[0]):
    alpha0 = a0[j] * beta[j]
    rxns0 = define_reactions(alpha0, beta[j], gamma[j])
    A0 = create_transition_matrix(rxns0, U_max, S_max)
    pi = stationary_from_transition_matrix(A0)
    pi_per_gene.append(pi)

In [251]:
# Calc forward state probabilities for each gene

X_fwd_list = []

for j in range(0, alpha.shape[0]):
    X_fwd = forward_distribution(A_per_gene[j], pi_per_gene[j], states, t, tau, state_grid)
    X_fwd_list.append(X_fwd)

In [366]:
Q.shape

(21325, 100)

In [368]:
# For each cell, calculate backward state probabilites

Q = traj.Q[:, 0, :] 
t_obs = np.argmax(Q[n,])

In [244]:
# Working cell index
n = 0
t_obs = max_t_indices[n]

In [260]:
# Working gene index
j = 0
A = A_per_gene[0]
X_fwd = X_fwd_list[0]

In [ ]:
u_curr = Y[n, :, 0]
s_curr = Y[n, :, 1]
x_curr = np.zeros(shape=(A[0].shape[0],), dtype="float")
x_curr[index_for[(u_curr, s_curr)]] = 1.0

In [355]:
def _reverse_generator(A, mu):
    diag_mu = np.diag(mu)
    diag_inv_mu = np.diag(1.0/mu)
    A_rev_off = diag_inv_mu @ A.T @ diag_mu
    np.fill_diagonal(A_rev_off, 0.0)
    A_rev = A_rev_off.copy()
    A_rev[np.diag_indices_from(A_rev)] = -A_rev_off.sum(axis=1) # Diagonals must be -sum(row)
    return A_rev

# Initialize distribution with observed counts
u_curr = np.round(Y[n, :, 0])[0]
s_curr = np.round(Y[n, :, 1])[0]
x_curr = np.zeros(shape=(A[0].shape[0],), dtype="float")
x_curr[index_for[(u_curr, s_curr)]] = 1.0

X_bw = np.zeros(shape=(len(states), len(t))) 
X_bw[:, t_obs] = x_curr

for k in reversed(range(1, t_obs + 1)):
    t_prev, t_curr = t[k-1], t[k]
    state_prev, state_curr = state_grid[k-1], state_grid[k]

    x_curr = X_bw[:, k]
    mu_k = X_fwd[:, k]
        
    if state_prev == state_curr:
        dt = t_curr - t_prev
        A_k = A[state_curr]
        A_k_rev =_reverse_generator(A_k, mu_k) 
        x_prev = x_curr @ sp.linalg.expm(A_k_rev * dt)

    else:
        # State switch happens in current interval             
        t_s = tau[state_curr]

        # Split backward march into 2 steps
        dt2 = t_curr - t_s # right interval: (state_switch_time, t_k]
        A_k2 = A[state_curr]
        A_k2_rev =_reverse_generator(A_k2, mu_k)
        x_mid = x_curr @ sp.linalg.expm(A_k2_rev * dt2) 
        
        dt1 = t_s - t_prev # left interval: (t_{k-1}, state_switch_time]
        A_k1 = A[state_prev]
        A_k1_rev =_reverse_generator(A_k1, mu_k) 
        x_prev = x_mid @ sp.linalg.expm(A_k1_rev * dt1) 
        
    X_bw[:, k-1] = x_prev

In [ ]:
    ###########################################################################
    
    print("Jump backwards from t =", np.round(t_curr, 3), "to t =", np.round(t_prev, 3), ":")
    print("Highest probability state (# unspliced, spliced):", states[np.argmax(x_curr)])
    idx = np.searchsorted(t, t_prev)
    print("Simulated (# unspliced, spliced):", np.round(Y[idx, :, 0], 2), np.round(Y[idx, :, 1], 2))
    print("Sum X(t_k) =", np.sum(x_curr))
    print("-------") 
    

In [ ]:
# Y_observed, Y, theta, rd, true_t, true_l = simulate_RNA(topo, tau, theta[0, :][None, :], n=20000, random_seed=666)